# Poster Equity Comparisons (SPY benchmark)

This notebook builds **combined equity-curve charts** (with SPY) for your poster:

1) **Logistic Regression, 3D**: F1→F5 + SPY in ONE chart (highlight F5)
2) **F5 only**: Logistic Regression 1D vs 3D + SPY in ONE chart

It recomputes the backtest curves from the same pipeline logic (time split + threshold tuned on val), so you don’t need to stitch PNGs.

Prereq: install deps and ensure `yfinance` works.
Run from `fyp_finance_ml_v2` root (recommended).


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import AppConfig
from src.data_loader import load_data
from src.features import build_feature_frame, feature_columns_for_set
from src.labels import add_forward_labels
from src.leakage import leakage_guard
from src.models import build_models, choose_threshold
from src.splits import time_split
from src.backtest import run_benchmark_buy_hold, run_top_k_execution_backtest

HERE = Path.cwd()
ROOT = HERE if (HERE / 'outputs').exists() else HERE.parent
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

config = AppConfig()
print('ROOT =', ROOT)
print('Will save figures to:', OUT_FIG)


## Helper: compute strategy daily net returns for one (feature_set, model, horizon)
Uses execution-aware convention: signal at t → enter open(t+1) → exit open(t+1+h).
Rebalance every h days.

In [ ]:
def compute_strategy_curve(*, feature_set: str, model_name: str, horizon: int, mode: str = 'live'):
    # Load data
    prices, benchmark, macro, fundamentals = load_data(config, mode=mode)
    prices = prices.loc[prices['ticker'].isin(config.tickers)].copy()
    if prices.empty:
        raise ValueError('No price rows after loading/filtering tickers')

    # Build features + labels
    feature_df = build_feature_frame(prices, benchmark, macro, fundamentals)
    full_df = add_forward_labels(feature_df, horizons=[horizon])

    label_col = f'label_{horizon}d'
    ret_col = f'fwd_ret_{horizon}d'

    groups = config.feature_sets[feature_set]
    feat_cols = feature_columns_for_set(groups)
    leakage_guard(feat_cols)

    keep_cols = list(dict.fromkeys(['date','ticker','open', label_col, ret_col] + feat_cols))
    work = full_df[keep_cols].copy()
    work = work.replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col, ret_col, 'open'])

    train, val, test, _ = time_split(work, config.train_frac, config.val_frac)
    valid_feat_cols = [c for c in feat_cols if c in train.columns and train[c].notna().any()]

    X_train = train[valid_feat_cols]
    y_train = train[label_col].astype(int)
    X_val = val[valid_feat_cols]
    y_val = val[label_col].astype(int)
    X_test = test[valid_feat_cols]

    if X_train.empty or X_val.empty or X_test.empty:
        raise ValueError('Empty split; not enough data after cleaning')

    models = build_models(config.random_seed, use_xgboost=config.use_xgboost)
    model = models[model_name]
    model.fit(X_train, y_train)

    val_proba = model.predict_proba(X_val)[:, 1]
    threshold, _ = choose_threshold(y_val, val_proba)

    test_proba = model.predict_proba(X_test)[:, 1]

    scored_test = test[['date','ticker','open', ret_col]].copy()
    scored_test['score'] = test_proba

    # execution-aware backtest
    daily, summary = run_top_k_execution_backtest(
        scored_test,
        score_col='score',
        open_col='open',
        horizon_days=horizon,
        top_k=config.top_k,
        transaction_cost_bps=config.transaction_cost_bps,
        rebalance_every=horizon,
    )
    daily = daily.copy()
    daily['equity'] = (1 + daily['net_ret']).cumprod()
    return daily, summary, benchmark


## Chart 1: Logistic Regression, 3D — F1→F5 + SPY (highlight F5)

In [ ]:
mode = 'live'
horizon = 3
model_name = 'logistic_regression'
feature_order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']

# Compute SPY benchmark equity on the same horizon
# (we reuse benchmark from first strategy call to avoid double downloads)
curves = {}
bench_daily = None

for fs in feature_order:
    daily, summary, benchmark = compute_strategy_curve(feature_set=fs, model_name=model_name, horizon=horizon, mode=mode)
    curves[fs] = daily
    if bench_daily is None:
        bench_daily, _ = run_benchmark_buy_hold(benchmark, horizon=horizon)
        bench_daily = bench_daily.copy()
        bench_daily['equity'] = (1 + bench_daily['net_ret']).cumprod()

plt.figure(figsize=(12, 6))
# SPY
plt.plot(pd.to_datetime(bench_daily['date']), bench_daily['equity'], label='SPY (Benchmark)', color='black', linewidth=2)

for fs, d in curves.items():
    lw = 3 if fs == 'F5_full_finance' else 1.5
    alpha = 1.0 if fs == 'F5_full_finance' else 0.75
    plt.plot(pd.to_datetime(d['date']), d['equity'], label=fs, linewidth=lw, alpha=alpha)

plt.title('Logistic Regression (3D): Equity Curves by Feature Set vs SPY (highlight F5)')
plt.xlabel('Date')
plt.ylabel('Equity (cumprod)')
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
out_path = OUT_FIG / 'poster_equity_logreg_3d_F1_to_F5_vs_SPY.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)


## Chart 2: F5 only — Logistic Regression 1D vs 3D + SPY

In [ ]:
mode = 'live'
feature_set = 'F5_full_finance'
model_name = 'logistic_regression'

d1, s1, benchmark = compute_strategy_curve(feature_set=feature_set, model_name=model_name, horizon=1, mode=mode)
bench1, _ = run_benchmark_buy_hold(benchmark, horizon=1)
bench1 = bench1.copy(); bench1['equity'] = (1 + bench1['net_ret']).cumprod()

d3, s3, benchmark = compute_strategy_curve(feature_set=feature_set, model_name=model_name, horizon=3, mode=mode)
bench3, _ = run_benchmark_buy_hold(benchmark, horizon=3)
bench3 = bench3.copy(); bench3['equity'] = (1 + bench3['net_ret']).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(pd.to_datetime(bench1['date']), bench1['equity'], label='SPY 1D', color='black', linewidth=1.5, alpha=0.5)
plt.plot(pd.to_datetime(bench3['date']), bench3['equity'], label='SPY 3D', color='black', linewidth=2)
plt.plot(pd.to_datetime(d1['date']), d1['equity'], label='F5 + Logistic 1D', color='#ef4444', linewidth=2, alpha=0.9)
plt.plot(pd.to_datetime(d3['date']), d3['equity'], label='F5 + Logistic 3D', color='#2563eb', linewidth=3)

plt.title('F5 + Logistic: 1D vs 3D Equity Curves vs SPY')
plt.xlabel('Date')
plt.ylabel('Equity (cumprod)')
plt.legend(fontsize=9)
plt.tight_layout()
out_path = OUT_FIG / 'poster_equity_F5_logreg_1D_vs_3D_vs_SPY.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

print('F5+Logistic 1D summary:', {k: round(float(v),4) for k,v in s1.items() if k in ['sharpe','max_drawdown','cumulative_return','annualized_return']})
print('F5+Logistic 3D summary:', {k: round(float(v),4) for k,v in s3.items() if k in ['sharpe','max_drawdown','cumulative_return','annualized_return']})
